In [ ]:
import os
import shutil
import random
from pathlib import Path
from typing import List, Tuple, Dict, Optional

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def _is_image(p: Path) -> bool:
    return p.suffix.lower() in IMAGE_EXTS


def _read_text(p: Path) -> str:
    return p.read_text(encoding="utf-8", errors="ignore")


def _is_yoloseg_label_empty(label_path: Path) -> bool:
    """A YOLO-seg label is 'empty' if it has no non-whitespace lines (or missing)."""
    if not label_path.exists():
        return True
    txt = _read_text(label_path).strip()
    if not txt:
        return True
    lines = [ln.strip() for ln in txt.splitlines() if ln.strip()]
    return len(lines) == 0


def _rewrite_to_single_class(label_in: Path, label_out: Path) -> None:
    """
    Convert multi-class YOLO-seg labels to 1-class (class_id=0),
    preserving polygons and multiple instances.
    """
    label_out.parent.mkdir(parents=True, exist_ok=True)

    if not label_in.exists():
        label_out.write_text("", encoding="utf-8")
        return

    lines = [ln.strip() for ln in _read_text(label_in).splitlines() if ln.strip()]
    if not lines:
        label_out.write_text("", encoding="utf-8")
        return

    new_lines = []
    for ln in lines:
        parts = ln.split()
        if len(parts) < 3:
            continue
        new_lines.append("0 " + " ".join(parts[1:]))

    label_out.write_text("\n".join(new_lines) + ("\n" if new_lines else ""), encoding="utf-8")


def _pair_images_labels(
    input_dir: Path,
    images_subdir: Optional[str] = None,
    labels_subdir: Optional[str] = None,
) -> List[Tuple[Path, Path]]:
    """
    Pair images to labels by stem.
    If images_subdir/labels_subdir are provided, looks there; otherwise searches input_dir recursively.
    """
    if images_subdir:
        img_root = input_dir / images_subdir
        images = [p for p in img_root.rglob("*") if p.is_file() and _is_image(p)]
    else:
        images = [p for p in input_dir.rglob("*") if p.is_file() and _is_image(p)]

    pairs = []
    for img in images:
        stem = img.stem
        if labels_subdir:
            lbl = (input_dir / labels_subdir / f"{stem}.txt")
        else:
            # Prefer sibling labels/ if present, else same dir
            lbl1 = img.parent.parent / "labels" / f"{stem}.txt"
            lbl2 = img.parent / f"{stem}.txt"
            lbl = lbl1 if lbl1.exists() else lbl2
            if not lbl.exists():
                matches = list(input_dir.rglob(f"{stem}.txt"))
                lbl = matches[0] if matches else lbl2  # may not exist

        pairs.append((img, lbl))

    return pairs


def split_balance_and_convert_to_one_class_3way(
    input_dir: str,
    output_dir: str,
    train_frac: float = 0.7,
    val_frac: float = 0.15,
    test_frac: float = 0.15,
    seed: int = 0,
    images_subdir: Optional[str] = None,
    labels_subdir: Optional[str] = None,
    copy_images: bool = True,
    balance_empty_ratio: float = 0.5,
) -> Dict[str, int]:
    """
    Produces a 3-way split for YOLO segmentation:

      output_dir/
        images/{train,val,test}
        labels/{train,val,test}

    Also:
      - Converts labels to 1-class (class_id=0), preserving multiple instances.
      - Downsamples empty-label images to target ~balance_empty_ratio empties overall.
        Default 0.5 -> ~50/50 empty vs non-empty.

    Notes:
      - Non-empty examples are always retained.
      - Empty examples are downsampled (never upsampled).
      - Splits are stratified by empty vs non-empty to keep ratios similar across splits.
    """
    # Validate split fractions
    total = train_frac + val_frac + test_frac
    if abs(total - 1.0) > 1e-6:
        raise ValueError(f"train_frac + val_frac + test_frac must equal 1.0 (got {total})")
    if not (0.0 < train_frac < 1.0 and 0.0 < val_frac < 1.0 and 0.0 < test_frac < 1.0):
        raise ValueError("train/val/test fractions must each be between 0 and 1 (exclusive).")

    in_dir = Path(input_dir)
    out_dir = Path(output_dir)

    out_img_train = out_dir / "images" / "train"
    out_img_val   = out_dir / "images" / "val"
    out_img_test  = out_dir / "images" / "test"

    out_lbl_train = out_dir / "labels" / "train"
    out_lbl_val   = out_dir / "labels" / "val"
    out_lbl_test  = out_dir / "labels" / "test"

    for p in [out_img_train, out_img_val, out_img_test, out_lbl_train, out_lbl_val, out_lbl_test]:
        p.mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)

    pairs = _pair_images_labels(in_dir, images_subdir=images_subdir, labels_subdir=labels_subdir)
    if not pairs:
        raise ValueError(f"No images found under: {input_dir}")

    empty_pairs: List[Tuple[Path, Path]] = []
    nonempty_pairs: List[Tuple[Path, Path]] = []
    missing_label = 0

    for img, lbl in pairs:
        if not lbl.exists():
            missing_label += 1
        if _is_yoloseg_label_empty(lbl):
            empty_pairs.append((img, lbl))
        else:
            nonempty_pairs.append((img, lbl))

    # Balance empties by downsampling to reach target ratio:
    # Want empty / (empty + nonempty) ~= balance_empty_ratio
    # => empty_target ~= (ratio / (1-ratio)) * nonempty
    if balance_empty_ratio <= 0 or balance_empty_ratio >= 1:
        raise ValueError("balance_empty_ratio must be between 0 and 1 (exclusive).")

    max_empty_to_keep = int(round((balance_empty_ratio / (1.0 - balance_empty_ratio)) * len(nonempty_pairs)))
    empty_target = min(len(empty_pairs), max_empty_to_keep)

    rng.shuffle(empty_pairs)
    empty_pairs_bal = empty_pairs[:empty_target]

    # Stratified 3-way split for nonempty and empty separately, then merge.
    def _split_3way(items: List[Tuple[Path, Path]]) -> Tuple[List[Tuple[Path, Path]], List[Tuple[Path, Path]], List[Tuple[Path, Path]]]:
        rng.shuffle(items)
        n = len(items)
        n_train = int(round(train_frac * n))
        n_val = int(round(val_frac * n))
        # ensure totals don't exceed n due to rounding
        n_train = min(n_train, n)
        n_val = min(n_val, n - n_train)
        n_test = n - n_train - n_val
        train = items[:n_train]
        val = items[n_train:n_train + n_val]
        test = items[n_train + n_val:]
        assert len(train) + len(val) + len(test) == n
        return train, val, test

    train_non, val_non, test_non = _split_3way(nonempty_pairs)
    train_emp, val_emp, test_emp = _split_3way(empty_pairs_bal)

    train_set = train_non + train_emp
    val_set   = val_non + val_emp
    test_set  = test_non + test_emp

    rng.shuffle(train_set)
    rng.shuffle(val_set)
    rng.shuffle(test_set)

    def _emit(split: List[Tuple[Path, Path]], img_out_root: Path, lbl_out_root: Path):
        for img, lbl in split:
            dst_img = img_out_root / img.name
            dst_lbl = lbl_out_root / f"{img.stem}.txt"

            if copy_images:
                shutil.copy2(img, dst_img)
            else:
                if dst_img.exists():
                    dst_img.unlink()
                os.symlink(str(img), str(dst_img))

            _rewrite_to_single_class(lbl, dst_lbl)

    _emit(train_set, out_img_train, out_lbl_train)
    _emit(val_set,   out_img_val,   out_lbl_val)
    _emit(test_set,  out_img_test,  out_lbl_test)

    def _count_empty(label_dir: Path) -> int:
        return sum(1 for p in label_dir.glob("*.txt") if _is_yoloseg_label_empty(p))

    stats = {
        "found_images": len(pairs),
        "missing_label_files_treated_as_empty": missing_label,
        "original_empty": len(empty_pairs),
        "original_nonempty": len(nonempty_pairs),

        "kept_empty": len(empty_pairs_bal),
        "kept_nonempty": len(nonempty_pairs),
        "kept_total": len(train_set) + len(val_set) + len(test_set),

        "train_images": len(train_set),
        "val_images": len(val_set),
        "test_images": len(test_set),

        "train_empty_labels": _count_empty(out_lbl_train),
        "val_empty_labels": _count_empty(out_lbl_val),
        "test_empty_labels": _count_empty(out_lbl_test),
    }

    stats["train_nonempty_labels"] = stats["train_images"] - stats["train_empty_labels"]
    stats["val_nonempty_labels"]   = stats["val_images"]   - stats["val_empty_labels"]
    stats["test_nonempty_labels"]  = stats["test_images"]  - stats["test_empty_labels"]

    return stats


# -----------------------------
# Example usage
# -----------------------------
stats = split_balance_and_convert_to_one_class_3way(
    input_dir=r"D:\AGAR_dataset\AGAR_dataset\yolo_dataset_big",
    output_dir=r"D:\AGAR_dataset\AGAR_dataset\yolo_dataset_big_split_1class_3way",
    train_frac=0.8,
    val_frac=0.1,
    test_frac=0.1,
    seed=42,
    balance_empty_ratio=0.5,  # target ~50/50 empty vs non-empty overall
)
print(stats)


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt")  # pick a pretrained *-seg checkpoint
results = model.train(
    data=r"D:\AGAR_dataset\AGAR_dataset\yolo_dataset_big_split_1class_3way\data.yaml",
    imgsz=640,
    epochs=100,
    batch=32,
    device=0,        # 0 for first GPU, or "cpu"
)


New https://pypi.org/project/ultralytics/8.4.0 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.203  Python-3.11.13 torch-2.8.0+cu126 CUDA:0 (Quadro P5000, 16384MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\AGAR_dataset\AGAR_dataset\yolo_dataset_split_1class\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, nam

In [2]:
import torch

In [3]:
torch.cuda.is_available()

True

In [ ]:
pip install ultralytics
pip install --force-reinstall torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu116
pip install albumentations
